# Week 7 Assignment — Regression, Classification & Evaluation

**Introduction to Data Science**

This assignment goes a bit further than the lab: fewer hints, more decisions for you to make.
Each task has a `# TODO` cell, followed by a **✅ Solution** cell — try it yourself first.

Dataset: `data/employee_data_cleaned.csv` (`employee_id`, `attendance`, `hours_worked`, `department`, `score`).

In [ ]:
import pandas as pd
df = pd.read_csv("data/employee_data_cleaned.csv")
df.head()

## Task 1 — A different single feature

So far the lecture and lab only used `attendance` to predict `score`. This dataset also has `hours_worked`.

**Your task:** train a `LinearRegression` using `hours_worked` alone as the feature, evaluate it (MAE, MSE, R²), and compare its MAE to the `attendance`-only model's MAE (build that one too, using the same split settings as the lecture: `test_size=0.2, random_state=42`).

In [ ]:
# TODO: train two single-feature regression models and compare their MAE


#### ✅ Solution

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Model A: attendance -> score
XA = df[["attendance"]]
y = df["score"]
XA_train, XA_test, y_train, y_test = train_test_split(XA, y, test_size=0.2, random_state=42)
model_a = LinearRegression().fit(XA_train, y_train)
pred_a = model_a.predict(XA_test)

# Model B: hours_worked -> score
XB = df[["hours_worked"]]
XB_train, XB_test, y_train2, y_test2 = train_test_split(XB, y, test_size=0.2, random_state=42)
model_b = LinearRegression().fit(XB_train, y_train2)
pred_b = model_b.predict(XB_test)

print("Attendance-only  MAE:", mean_absolute_error(y_test, pred_a), " R2:", r2_score(y_test, pred_a))
print("Hours-worked-only MAE:", mean_absolute_error(y_test2, pred_b), " R2:", r2_score(y_test2, pred_b))

**Discussion (sample answer):** because `y` and the train/test indices come from the same `random_state=42` split each time, the two models are being judged on the same rows. Whichever feature has the lower MAE (and higher R²) is doing more of the real work in explaining `score` — in this dataset, `attendance` should come out noticeably ahead, since it's the feature that was built to correlate strongly with `score`.

## Task 2 — Multiple features together

**Your task:** train a `LinearRegression` using **both** `attendance` and `hours_worked` as features. Does adding the second feature improve MAE/R² over the attendance-only model from Task 1? Print the two coefficients and say which feature the model is relying on more.

In [ ]:
# TODO


#### ✅ Solution

In [ ]:
X_multi = df[["attendance", "hours_worked"]]
X_multi_train, X_multi_test, y_train3, y_test3 = train_test_split(X_multi, y, test_size=0.2, random_state=42)

model_multi = LinearRegression().fit(X_multi_train, y_train3)
pred_multi = model_multi.predict(X_multi_test)

print("Multi-feature MAE:", mean_absolute_error(y_test3, pred_multi), " R2:", r2_score(y_test3, pred_multi))
print("Coefficients (attendance, hours_worked):", model_multi.coef_)
print("Intercept:", model_multi.intercept_)

**Discussion (sample answer):** compare this MAE/R² to the attendance-only model in Task 1 (same split, so it's a fair comparison). If R² barely moves and the `hours_worked` coefficient is small relative to the `attendance` coefficient, that tells you `hours_worked` isn't adding much new information beyond what `attendance` already captures — a smaller MAE improvement isn't automatically worth added model complexity.

## Task 3 — Move the classification threshold

The lecture defined `high_performer` as `score >= 80`. That's a strict bar.

**Your task:** create a new target `high_performer_75` using `score >= 75` instead. Train a `DecisionTreeClassifier(random_state=42)` on `attendance` to predict it (same `test_size=0.2, random_state=42` split settings), and print accuracy, precision, recall, and F1 (you can get all four from `classification_report`, or compute precision/recall manually from the confusion matrix like in the lecture).

Then answer: compared to the `score >= 80` version from the lab, would you expect precision to go up or down, and why?

In [ ]:
# TODO


#### ✅ Solution

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

df["high_performer_75"] = df["score"] >= 75

Xc = df[["attendance"]]
yc75 = df["high_performer_75"]
Xc_train, Xc_test, yc75_train, yc75_test = train_test_split(Xc, yc75, test_size=0.2, random_state=42)

clf75 = DecisionTreeClassifier(random_state=42)
clf75.fit(Xc_train, yc75_train)
preds75 = clf75.predict(Xc_test)

print("Accuracy:", accuracy_score(yc75_test, preds75))
print(confusion_matrix(yc75_test, preds75))
print(classification_report(yc75_test, preds75))

**Discussion (sample answer):** lowering the bar from 80 to 75 means more employees now count as "high performers," so the positive class is less rare/less strict. In general, a looser, more inclusive threshold tends to make it *easier* to catch true positives (recall often goes up) but *easier* to misclassify a borderline case as positive too (precision can go down) — the exact numbers depend on where the data actually clusters, so always check the printed report rather than assuming.

## Task 4 — Model comparison at the new threshold

**Your task:** using the `high_performer_75` target and the same train/test split from Task 3, train both `LogisticRegression()` and `DecisionTreeClassifier(random_state=42)`, and print accuracy for each. Which one wins here? Is it the same winner as in the lab (with the `>= 80` threshold)?

In [ ]:
# TODO


#### ✅ Solution

In [ ]:
from sklearn.linear_model import LogisticRegression

models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

for name, m in models.items():
    m.fit(Xc_train, yc75_train)
    preds = m.predict(Xc_test)
    print(name, "->", "accuracy:", accuracy_score(yc75_test, preds))

**Discussion (sample answer):** it's entirely possible for the "winning" model to flip depending on the threshold used to define the target — that's a useful reminder that "which model is better" isn't a fixed fact about the algorithms, it depends on the exact problem (including how you define your labels) and the data split. This is also a preview of why the paid course covers cross-validation: one split, one threshold isn't enough to declare a permanent winner.

## Task 5 — Baseline sanity check (both tasks)

**Your task:** for the Task 2 regression model AND the Task 3 classification model, compute the appropriate baseline (mean-prediction MAE for regression, most-common-class accuracy for classification) and state in one sentence each whether the trained model actually beat its baseline.

In [ ]:
# TODO


#### ✅ Solution

In [ ]:
# Regression baseline (Task 2's split: y_train3 / y_test3)
baseline_pred = y_train3.mean()
baseline_mae = (y_test3 - baseline_pred).abs().mean()
print("Regression baseline MAE:", baseline_mae, " | Multi-feature model MAE:", mean_absolute_error(y_test3, pred_multi))

# Classification baseline (Task 3's split: yc75_train / yc75_test)
baseline_class = yc75_train.mode()[0]
baseline_acc = (yc75_test == baseline_class).mean()
print("Classification baseline accuracy:", baseline_acc, " | Decision Tree accuracy:", accuracy_score(yc75_test, preds75))

**Sample statements:**
- *Regression:* the multi-feature model's MAE is lower than the baseline's MAE, so it is learning a real, useful relationship rather than just repeating the average.
- *Classification:* the Decision Tree's accuracy is higher than always guessing the majority class, so it's doing better than "doing nothing clever" — though it's worth double-checking precision/recall too, since accuracy alone can be misleading on an imbalanced target.

*(Your printed numbers may differ slightly depending on how you built each split — the direction of the comparison is what matters.)*

---
### Submission checklist
- [ ] All TODO cells filled in with your own code before checking the solution
- [ ] Every cell runs top-to-bottom without errors
- [ ] Written discussion answers reflect **your own** printed numbers, not just the sample answers above
